In [ ]:
import tensorflow as tf
import numpy as np
import gym
from ou_noise import OUNoise
import random


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })



def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.0001
    CRITIC_LEARNING_RATE =  0.001
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v1'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
    epochs = 15000
#     epochs = 20000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = DDPG(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        llegadas =0
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0 
            max_steps = 45 #es solo a efecto de contador
            r_episodio_actual = []
            #llegadas =0
            while (not done):
                #print('step =', step)
                #wait = input("PRESS ENTER TO CONTINUE.")
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)
                    #print('Explorando')

                action = np.clip(action,min_action,max_action)
                #print('ruido =', max(epsilon,0)*ruido.noise(),'epsilon =',epsilon)
                #print('LA ACCION clipeada ES', action, action.shape)
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                #print('Steps = ', info['Steps'],'V0 =',info['v0'])
                #print('normalized_state =',normalized_state)
                #wait = input("PRESS ENTER TO CONTINUE.")
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
#                 print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
#                 print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                
                step += 1
            
            
            #if info['Steps'] > max_steps:
             #   llegadas +=1
#             print ('--------------------------------------------')
#             print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
#             print ('--------------------------------------------')

                

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_DDPG01.npy',Reward_episodios)
            

            print('epoch          = ', i)      
            print('Episode Reward = ', episode_r/10)
            print('Last Reward    = ', reward)
            print('State          = ', state)
            print()

            if i % 1000 == 0:      # Save model every 1000 iterations
                ddpg.save()
            
        ddpg.save()

C:\Users\archi\anaconda3\envs\BTP\lib\site-packages\gym\logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


epoch          =  0
Episode Reward =  51.785921000000016
Last Reward    =  0.39935
State          =  [   48.002 17971.473   623.83 ]

Model saved in file: actor_model
epoch          =  1
Episode Reward =  66.49910600000001
Last Reward    =  0.0024
State          =  [ 151.454   1847.1482  -272.46344]

epoch          =  2
Episode Reward =  66.59231100000001
Last Reward    =  0.29841
State          =  [  85.241   3124.3416   -58.86363]

epoch          =  3
Episode Reward =  35.98161600000003
Last Reward    =  0.27097
State          =  [   43.372  16266.649   1205.2251]

epoch          =  4
Episode Reward =  64.09188200000003
Last Reward    =  0.27523
State          =  [   47.474  11711.8125 -1644.8345]

epoch          =  5
Episode Reward =  62.07429200305597
Last Reward    =  0.0
State          =  [  207.043      0.     -6606.0034]

epoch          =  6
Episode Reward =  74.8807307747117
Last Reward    =  0.0
State          =  [159.446   0.      0.   ]

epoch          =  7
Episode Reward =

c:\users\archi\python code\btp\luis_upgrade\mppt-gym_v4\gym_mppt\envs\mppt_env_shaded1.py:39: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)


epoch          =  41
Episode Reward =  74.50210688849126
Last Reward    =  0.0
State          =  [206.706   0.      0.   ]

epoch          =  42
Episode Reward =  51.80065512202428
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  43
Episode Reward =  46.79803501201389
Last Reward    =  0.3111000061035156
State          =  [   44.659 10714.045 -1171.291]

epoch          =  44
Episode Reward =  54.137667912424924
Last Reward    =  0.08787
State          =  [   28.905  10850.535  -1325.0648]

epoch          =  45
Episode Reward =  61.47049219168997
Last Reward    =  0.00291
State          =  [   3.583   1345.3301   455.17557]

epoch          =  46
Episode Reward =  62.666103107371875
Last Reward    =  0.11382
State          =  [ 107.77   3744.2756 -101.551 ]

epoch          =  47
Episode Reward =  69.74677616025207
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  48
Episode Reward =  60.288118999999845
Last Reward    =  0.07703
State          =  [

epoch          =  100
Episode Reward =  75.8205731205817
Last Reward    =  0.0
State          =  [ 212.         0.      -685.41394]

epoch          =  101
Episode Reward =  39.23714899421091
Last Reward    =  0.4863
State          =  [   76.769   28680.246    -130.55309]

epoch          =  102
Episode Reward =  32.140498960659286
Last Reward    =  0.42298999428749084
State          =  [   70.442  26359.91    2601.8516]

epoch          =  103
Episode Reward =  43.36535680990535
Last Reward    =  0.06853
State          =  [  18.617   6980.0063   845.72406]

epoch          =  104
Episode Reward =  64.22553700960137
Last Reward    =  0.9860299825668335
State          =  [7.2811996e+01 2.3501303e+04 2.4414062e+00]

epoch          =  105
Episode Reward =  70.52055678995573
Last Reward    =  0.05877
State          =  [   17.691   6634.597  -1142.4456]

epoch          =  106
Episode Reward =  58.6283449769211
Last Reward    =  0.5593799948692322
State          =  [   93.145 34723.594  2254.207

...

epoch          =  9942
Episode Reward =  11.947488999685962
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  9943
Episode Reward =  12.238261947907587
Last Reward    =  0.03638
State          =  [  10.794  4051.5342 1335.2902]

epoch          =  9944
Episode Reward =  15.059753998900527
Last Reward    =  0.0
State          =  [  0.        0.      -58.49147]

epoch          =  9945
Episode Reward =  15.152459001111714
Last Reward    =  0.00323
State          =  [    1.707    641.081  -1103.3145]

epoch          =  9946
Episode Reward =  11.918143000055192
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  9947
Episode Reward =  14.550818999333558
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  9948
Episode Reward =  10.078076999779954
Last Reward    =  0.03152
State          =  [  18.983  7126.82   2616.2559]

epoch          =  9949
Episode Reward =  16.934405152558988
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch

epoch          =  10009
Episode Reward =  14.893428998693746
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10010
Episode Reward =  11.837280999939768
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10011
Episode Reward =  10.628145999821347
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10012
Episode Reward =  11.945078000166243
Last Reward    =  0.00728
State          =  [   8.1    3041.1257 3041.1257]

epoch          =  10013
Episode Reward =  14.18745799951554
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10014
Episode Reward =  19.44033900476446
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10015
Episode Reward =  12.468147002277231
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10016
Episode Reward =  18.815136000165225
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10017
Episode Reward =  15.600661997138484
Last Reward 

epoch          =  10076
Episode Reward =  11.952658999677098
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10077
Episode Reward =  9.000252000000577
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10078
Episode Reward =  14.725114999370112
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10079
Episode Reward =  15.083817000202027
Last Reward    =  0.00013
State          =  [  0.892   335.09302 335.09302]

epoch          =  10080
Episode Reward =  10.677221999886077
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10081
Episode Reward =  11.953854000031388
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10082
Episode Reward =  10.640056999867692
Last Reward    =  0.02906
State          =  [  16.107  6047.1484 3718.3792]

epoch          =  10083
Episode Reward =  17.10292400035677
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10084
Episode Reward =  15.012

epoch          =  10142
Episode Reward =  13.424381021612973
Last Reward    =  0.00588
State          =  [   6.622 2486.462 2486.462]

epoch          =  10143
Episode Reward =  14.630036999645029
Last Reward    =  0.0
State          =  [   0.         0.      -466.18573]

epoch          =  10144
Episode Reward =  13.310993916097072
Last Reward    =  0.04533
State          =  [  23.448  8803.274  1120.1107]

epoch          =  10145
Episode Reward =  13.955274999987974
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10146
Episode Reward =  18.227479920475712
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10147
Episode Reward =  19.200998971941143
Last Reward    =  0.16686
State          =  [  13.052  4898.583  1965.1042]

epoch          =  10148
Episode Reward =  8.996875999772099
Last Reward    =  0.04453999921679497
State          =  [  16.933  5935.486  2848.0986]

epoch          =  10149
Episode Reward =  11.061996913204467
Last Reward    =

epoch          =  10208
Episode Reward =  14.974085999912154
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10209
Episode Reward =  19.297868960464747
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10210
Episode Reward =  16.773993000150416
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10211
Episode Reward =  10.611470000027365
Last Reward    =  0.0
State          =  [    0.         0.     -3018.7126]

epoch          =  10212
Episode Reward =  13.401931999987351
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10213
Episode Reward =  14.464242999428752
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10214
Episode Reward =  10.972881000036953
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10215
Episode Reward =  18.451000893403993
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  10216
Episode Reward =  15.148450000663399
Last Reward

In [ ]:
# 7500  -> 100 minutes (1 hour 40 mins)
# 15000 -> 200 minutes (3 hours 20 mins) # X around 9 hours